In [4]:
import pandas as pd
ratings = pd.read_csv("../data/ml-latest-small/ratings.csv")
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
print("Ratings:", len(ratings))
print("Users:", ratings["userId"].nunique())
print("Movies rated:", ratings["movieId"].nunique())

print(ratings["rating"].describe())


Ratings: 100836
Users: 610
Movies rated: 9724
count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64


In [7]:
ratings.isnull().sum()
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [8]:
# Given a user and a movie, predict how much that user would like that movie.

# need to split the data intially

from sklearn.model_selection import train_test_split

train, test = train_test_split(
    ratings,
    test_size = 0.2,
    random_state = 42
)

In [9]:
# making sure that I am in the correct env
import sys
print(sys.executable)

/opt/homebrew/anaconda3/envs/movie-recommender/bin/python


In [10]:
# getting the unique users and movies

user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

num_users = len(user_ids)
num_movies = len(movie_ids)

print("Users:", num_users)
print("Movies:", num_movies)

Users: 610
Movies: 9724


In [11]:
# converting IDs into mappings for the model
user_to_index = {user_id: i for i, user_id in enumerate(user_ids)}
movie_to_index = {movie_id: i for i, movie_id in enumerate(movie_ids)}

train["user_index"] = train["userId"].map(user_to_index)
train["movie_index"] = train["movieId"].map(movie_to_index)

test["user_index"] = test["userId"].map(user_to_index)
test["movie_index"] = test["movieId"].map(movie_to_index)


In [12]:
# build the reccomendation model

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [26]:
# class MovieRecommender(keras.Model):
#     def __init__(self, num_users, num_movies, embedding_size=50):
#         super().__init__()

#         self.user_embedding = layers.Embedding(
#             input_dim=num_users,
#             output_dim=embedding_size
#         )

#         self.movie_embedding = layers.Embedding(
#             input_dim=num_movies,
#             output_dim=embedding_size
#         )

#     def call(self, inputs):
#         user_vector = self.user_embedding(inputs[:, 0])
#         movie_vector = self.movie_embedding(inputs[:, 1])

#         return tf.reduce_sum(user_vector * movie_vector, axis=1)


# update class with new bias term
class MovieRecommender(keras.Model):
    def __init__(self, num_users, num_movies, embedding_size=50):
        super().__init__()

        # User and movie embeddings
        self.user_embedding = layers.Embedding(
            input_dim=num_users,
            output_dim=embedding_size
        )

        self.movie_embedding = layers.Embedding(
            input_dim=num_movies,
            output_dim=embedding_size
        )

        # Bias terms
        self.user_bias = layers.Embedding(
            input_dim=num_users,
            output_dim=1
        )

        self.movie_bias = layers.Embedding(
            input_dim=num_movies,
            output_dim=1
        )

    def call(self, inputs):
        user_vector = self.user_embedding(inputs[:, 0])
        movie_vector = self.movie_embedding(inputs[:, 1])

        user_bias = self.user_bias(inputs[:, 0])
        movie_bias = self.movie_bias(inputs[:, 1])

        interaction = tf.reduce_sum(
            user_vector * movie_vector,
            axis=1
        )

        return (
            interaction
            + tf.squeeze(user_bias, axis=1)
            + tf.squeeze(movie_bias, axis=1)
        )

In [14]:
# build the model

model = MovieRecommender(
    num_users=num_users,
    num_movies=num_movies
)

In [15]:
# compile model
model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)


In [16]:
# prepare training inputs

X_train = train[["user_index", "movie_index"]].values
y_train = train["rating"].values

X_test = test[["user_index", "movie_index"]].values
y_test = test["rating"].values

print(X_train[:5])
print(y_train[:5])

[[ 508 4285]
 [ 325 5629]
 [  56  134]
 [ 609   66]
 [ 461 1172]]
[3. 4. 3. 4. 2.]


In [17]:
# training

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)

Epoch 1/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 10.9627 - mean_absolute_error: 3.0843 - val_loss: 4.7608 - val_mean_absolute_error: 1.8375
Epoch 2/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 2.6408 - mean_absolute_error: 1.2691 - val_loss: 2.0385 - val_mean_absolute_error: 1.0680
Epoch 3/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.3184 - mean_absolute_error: 0.8647 - val_loss: 1.5712 - val_mean_absolute_error: 0.9079
Epoch 4/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.9454 - mean_absolute_error: 0.7381 - val_loss: 1.4190 - val_mean_absolute_error: 0.8533
Epoch 5/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.7719 - mean_absolute_error: 0.6727 - val_loss: 1.3577 - val_mean_absolute_error: 0.8282
Epoch 6/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6662 - mean_absolute_error: 0.6264 - val_loss: 1.3314 - val_mean_absolute_error: 0.8161
Epoch 7/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.5896 - mean_absolute_er

In [18]:
test_loss, test_mae = model.evaluate(X_test, y_test)

print("Test MAE:", test_mae)

631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 323us/step - loss: 1.3048 - mean_absolute_error: 0.8106
Test MAE: 0.8106088638305664


In [19]:
# model is trained. now want to start generating movie recs
# lets start with user 1

user_id = 1
user_ratings = ratings[ratings["userId"] == user_id]

user_ratings = user_ratings.merge(
    movies,
    on="movieId"
)

user_ratings.sort_values("rating", ascending=False).head(10)

,userId,movieId,rating,timestamp,title,genres
231,1,5060,5.0,964984002,M*A*S*H (a.k.a. MASH) (1970),Comedy|Drama|War
185,1,2872,5.0,964981680,Excalibur (1981),Adventure|Fantasy
89,1,1291,5.0,964981909,Indiana Jones and the Last Crusade (1989),Action|Adventure
90,1,1298,5.0,964984086,Pink Floyd: The Wall (1982),Drama|Musical
190,1,2948,5.0,964982191,From Russia with Love (1963),Action|Adventure|Thriller
189,1,2947,5.0,964982176,Goldfinger (1964),Action|Adventure|Thriller
188,1,2944,5.0,964981872,"Dirty Dozen, The (1967)",Action|Drama|War
186,1,2899,5.0,964982703,Gulliver's Travels (1939),Adventure|Animation|Children
184,1,2858,5.0,964980868,American Beauty (1999),Drama|Romance
179,1,2700,5.0,964980985,"South Park: Bigger, Longer and Uncut (1999)",Animation|Comedy|Musical


In [20]:
# finding movies 1 has not seen

watched_movies = set(user_ratings["movieId"])

unwatched_movies = movies[
    ~movies["movieId"].isin(watched_movies)
].copy()

print(unwatched_movies.head())

   movieId                               title                      genres
1        2                      Jumanji (1995)  Adventure|Children|Fantasy
3        4            Waiting to Exhale (1995)        Comedy|Drama|Romance
4        5  Father of the Bride Part II (1995)                      Comedy
6        7                      Sabrina (1995)              Comedy|Romance
7        8                 Tom and Huck (1995)          Adventure|Children


In [21]:
# convert IDs to model indicies -> keep only the movies that the model knows and shows up in rating data
unwatched_movies = unwatched_movies[
    unwatched_movies["movieId"].isin(movie_to_index)
].copy()

unwatched_movies["movie_index"] = (
    unwatched_movies["movieId"].map(movie_to_index)
)

In [22]:
# get user's model index
user_index = user_to_index[user_id]

In [23]:
# asking model to predict rating

import numpy as np

user_indices = np.full(
    len(unwatched_movies),
    user_index
)

movie_indices = unwatched_movies["movie_index"].values

prediction_input = np.column_stack(
    (user_indices, movie_indices)
)


In [24]:
# predictions added to movies 

predictions = model.predict(prediction_input)
unwatched_movies["predicted_rating"] = predictions

297/297 ━━━━━━━━━━━━━━━━━━━━ 0s 254us/step


In [25]:
# get top 10 recs
# milestone: model is actaully recommending movies

recommendations = unwatched_movies.sort_values(
    "predicted_rating",
    ascending=False
).head(10)

recommendations[
    ["title", "genres", "predicted_rating"]
]

,title,genres,predicted_rating
9472,Call Me by Your Name (2017),Drama|Romance,6.256853
8709,Delirium (2014),Adventure|Romance|Sci-Fi,6.185369
3505,Phantom of the Paradise (1974),Comedy|Fantasy|Horror|Musical|Thriller,6.136122
2504,Never Cry Wolf (1983),Adventure|Drama,6.078607
5037,Dune (2000),Drama|Fantasy|Sci-Fi,6.061872
9042,Deathgasm (2015),Comedy|Horror,6.032655
2963,"Legend of Drunken Master, The (Jui kuen II) (1...",Action|Comedy,6.027087
5489,"Holy Mountain, The (Montaña sagrada, La) (1973)",Drama,6.011631
813,Shall We Dance (1937),Comedy|Musical|Romance,5.992987
8272,Blue Is the Warmest Color (La vie d'Adèle) (2013),Drama|Romance,5.992172


In [27]:
# creating a new model to compare to the baseline model.
# new model includes bias terms -> user bias learns whether particular 
# user tends to rate movies higher or lower than other users.

model_bias = MovieRecommender(
    num_users=num_users,
    num_movies=num_movies
)

model_bias.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)

In [29]:
# train new model under same condtions
history_bias = model_bias.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1
)


Epoch 1/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 9.6134 - mean_absolute_error: 2.8559 - val_loss: 3.9526 - val_mean_absolute_error: 1.6629
Epoch 2/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 2.1374 - mean_absolute_error: 1.1416 - val_loss: 1.5796 - val_mean_absolute_error: 0.9583
Epoch 3/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.0650 - mean_absolute_error: 0.7869 - val_loss: 1.2159 - val_mean_absolute_error: 0.8246
Epoch 4/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.7843 - mean_absolute_error: 0.6770 - val_loss: 1.1050 - val_mean_absolute_error: 0.7815
Epoch 5/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6458 - mean_absolute_error: 0.6129 - val_loss: 1.0626 - val_mean_absolute_error: 0.7668
Epoch 6/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.5504 - mean_absolute_error: 0.5612 - val_loss: 1.0503 - val_mean_absolute_error: 0.7597
Epoch 7/10
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.4778 - mean_absolute_err

In [30]:
# find new MAE

test_loss, test_mae_bias = model_bias.evaluate(
    X_test,
    y_test
)

print("Baseline MAE:", test_mae)
print("Bias model MAE:", test_mae_bias)

631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 247us/step - loss: 1.0495 - mean_absolute_error: 0.7604
Baseline MAE: 0.8106088638305664
Bias model MAE: 0.7604140639305115


In [ ]:
# keeping the current model and bias unchanged so that I can retain for comparison
# creating new hybrid model

# converting genres into binary features

from sklearn.preprocessing import MultiLabelBinarizer

movies["genre_list"] = movies["genres"].str.split("|")

mlb = MultiLabelBinarizer()

genre_matrix = mlb.fit_transform(movies["genre_list"])

print(mlb.classes_)
print(genre_matrix.shape)

['(no genres listed)' 'Action' 'Adventure' 'Animation' 'Children' 'Comedy'
 'Crime' 'Documentary' 'Drama' 'Fantasy' 'Film-Noir' 'Horror' 'IMAX'
 'Musical' 'Mystery' 'Romance' 'Sci-Fi' 'Thriller' 'War' 'Western']
(9742, 20)


In [ ]:
# attatching genre feature to each rating

genre_df = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,
    index=movies["movieId"]
)

In [33]:
genre_df.loc[1]

(no genres listed)    0
Action                0
Adventure             1
Animation             1
Children              1
Comedy                1
Crime                 0
Documentary           0
Drama                 0
Fantasy               1
Film-Noir             0
Horror                0
IMAX                  0
Musical               0
Mystery               0
Romance               0
Sci-Fi                0
Thriller              0
War                   0
Western               0
Name: 1, dtype: int64

In [ ]:
# creating genre inputs for training/testing

X_train_genres = genre_df.loc[
    train["movieId"]
].values.astype("float32")

X_test_genres = genre_df.loc[
    test["movieId"]
].values.astype("float32")

In [ ]:
# checking the same number of rows
print(X_train.shape)
print(X_train_genres.shape)

(80668, 2)
(80668, 20)


In [79]:
# creating new model class

class HybridMovieRecommender(keras.Model):
    def __init__(self, num_users, num_movies, num_genres, embedding_size=50):
        super().__init__()

        self.user_embedding = layers.Embedding(
            num_users, embedding_size
        )

        self.movie_embedding = layers.Embedding(
            num_movies, embedding_size
        )

        self.user_bias = layers.Embedding(
            num_users, 1
        )

        self.movie_bias = layers.Embedding(
            num_movies, 1
        )

        # Converts genre information into the same
        # 50-dimensional space as the embeddings
        self.genre_dense = layers.Dense(
            embedding_size,
            activation="relu"
        )

    def call(self, inputs):

        user_indices = inputs["user"]
        movie_indices = inputs["movie"]
        genres = inputs["genres"]

        user_vector = self.user_embedding(user_indices)
        movie_vector = self.movie_embedding(movie_indices)

        user_bias = self.user_bias(user_indices)
        movie_bias = self.movie_bias(movie_indices)

        genre_vector = self.genre_dense(genres)

        # Combine the movie embedding with its genre information
        combined_movie = movie_vector + genre_vector

        interaction = tf.reduce_sum(
            user_vector * combined_movie,
            axis=1
        )

        return (
            global_mean
            + interaction
            + tf.squeeze(user_bias, axis=1)
            + tf.squeeze(movie_bias, axis=1)
        )

In [99]:
# creating new model

hybrid_model = HybridMovieRecommender(
    num_users=num_users,
    num_movies=num_movies,
    num_genres=X_train_genres.shape[1]
)

In [100]:
# compile

hybrid_model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)

In [101]:
# train

hybrid_X_train = {
    "user": X_train[:, 0],
    "movie": X_train[:, 1],
    "genres": X_train_genres
}

In [102]:
# early_stopping = keras.callbacks.EarlyStopping(
#     monitor="val_loss",
#     patience=2,
#     restore_best_weights=True
# )

history = hybrid_model.fit(
    hybrid_X_train,
    y_train,
    epochs=6,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping]
)

Epoch 1/6
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.8688 - mean_absolute_error: 0.7258 - val_loss: 0.8186 - val_mean_absolute_error: 0.7017
Epoch 2/6
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6980 - mean_absolute_error: 0.6401 - val_loss: 0.7549 - val_mean_absolute_error: 0.6715
Epoch 3/6
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 980us/step - loss: 0.5716 - mean_absolute_error: 0.5725 - val_loss: 0.7232 - val_mean_absolute_error: 0.6540
Epoch 4/6
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.4645 - mean_absolute_error: 0.5092 - val_loss: 0.7200 - val_mean_absolute_error: 0.6521
Epoch 5/6
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.3721 - mean_absolute_error: 0.4490 - val_loss: 0.7229 - val_mean_absolute_error: 0.6547
Epoch 6/6
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.2930 - mean_absolute_error: 0.3921 - val_loss: 0.7364 - val_mean_absolute_error: 0.6614


In [103]:
hybrid_X_test = {
    "user": X_test[:, 0],
    "movie": X_test[:, 1],
    "genres": X_test_genres
}

test_loss, hybrid_mae = hybrid_model.evaluate(
    hybrid_X_test,
    y_test
)

print("Baseline MAE:", test_mae)
print("Bias model MAE:", test_mae_bias)
print("Hybrid model MAE:", hybrid_mae)

631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 478us/step - loss: 0.7148 - mean_absolute_error: 0.6459
Baseline MAE: 0.8106088638305664
Bias model MAE: 0.7604140639305115
Hybrid model MAE: 0.645858645439148


In [ ]:
global_mean = y_train.mean()
print(global_mean)

631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 335us/step - loss: 0.7199 - mean_absolute_error: 0.6495
Learning rate 0.001 MAE: 0.64
Learning rate 0.0005 MAE: 0.6494518518447876
Epochs trained: 3
